# 22 · Indexing & Query Performance

Correct SQL that's slow still fails in production. This module teaches how the
engine finds rows and how to make it fast:
- reading `EXPLAIN QUERY PLAN` (`SCAN` vs `SEARCH`)
- single-column, **composite**, **covering**, and **unique** indexes
- when an index *won't* be used
- `ANALYZE` and the write-time cost of indexes

> Demo indexes are prefixed `demo_` and dropped at the end.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## `EXPLAIN QUERY PLAN`
It shows *how* SQLite will run a query. Two words to watch for:
- **`SCAN`** — read every row (a full table scan; fine for tiny tables, deadly
  for large ones).
- **`SEARCH ... USING INDEX`** — jump straight to matching rows via an index.

With no useful index, filtering `orders` by `customer_id` is a full scan:

In [ ]:
%%sql
EXPLAIN QUERY PLAN
SELECT * FROM orders WHERE customer_id = 1;

## Add an index, watch the plan change

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_orders_customer;
CREATE INDEX demo_idx_orders_customer ON orders(customer_id);
EXPLAIN QUERY PLAN
SELECT * FROM orders WHERE customer_id = 1;

The plan now reads `SEARCH ... USING INDEX demo_idx_orders_customer` — an indexed lookup instead of a scan.

## Composite indexes & the leftmost-prefix rule
An index on `(a, b)` can serve queries filtering on `a`, or `a AND b`, but **not
`b` alone** — like a phone book sorted by (last, first). Order the columns by how
you query them.

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_prod_cat_price;
CREATE INDEX demo_idx_prod_cat_price ON products(category_id, unit_price);
EXPLAIN QUERY PLAN
SELECT product_name FROM products WHERE category_id = 1 AND unit_price > 50;

## Covering index
If an index contains **every column a query needs**, SQLite answers from the
index alone and never touches the table — shown as `USING COVERING INDEX`.

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_cover;
CREATE INDEX demo_idx_cover ON products(category_id, unit_price, product_name);
EXPLAIN QUERY PLAN
SELECT category_id, unit_price, product_name
FROM products WHERE category_id = 2;

## Unique index
Enforces uniqueness *and* speeds lookups. (A `UNIQUE` constraint creates one
automatically — e.g. `customers.email`.)

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_unique_sku;
CREATE UNIQUE INDEX demo_idx_unique_sku ON products(product_name);
SELECT 'unique index created' AS status;

## ⚠️ When an index is *not* used
- **A function/expression on the column:** `WHERE LOWER(product_name) = 'x'`
  can't use a plain index on `product_name` (you'd need an *expression index*).
- **A leading wildcard:** `LIKE '%mouse'` can't seek; `LIKE 'mouse%'` can.
- **Low selectivity:** if a value matches most rows, a scan is actually cheaper.

Example the planner will still scan:

In [ ]:
%%sql
EXPLAIN QUERY PLAN
SELECT * FROM products WHERE LOWER(product_name) = 'usb-c hub';

## `ANALYZE` and the cost of indexes
`ANALYZE` gathers statistics so the planner makes better choices. Remember every
index **speeds reads but slows writes** (each `INSERT`/`UPDATE`/`DELETE` must
maintain it) and uses disk — index the columns your queries actually filter and
join on, not every column.

In [ ]:
%%sql
ANALYZE;
SELECT 'stats gathered' AS status;

## Clean up

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_orders_customer;
DROP INDEX IF EXISTS demo_idx_prod_cat_price;
DROP INDEX IF EXISTS demo_idx_cover;
DROP INDEX IF EXISTS demo_idx_unique_sku;
SELECT 'cleaned up' AS status;

## Practice

**✏️ Exercise 1.** Create an index on order_items(product_id), then use EXPLAIN QUERY PLAN to confirm a lookup by product_id uses it. Drop it afterwards.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_oi_product;
CREATE INDEX demo_idx_oi_product ON order_items(product_id);
EXPLAIN QUERY PLAN SELECT * FROM order_items WHERE product_id = 6;

### ✅ Recap
Read plans (`SCAN` = bad on big tables, `SEARCH USING INDEX` = good). Composite
indexes follow the leftmost-prefix rule; covering indexes avoid table lookups;
functions/leading-wildcards defeat indexes. Index deliberately — reads get
faster, writes get slower.

**Next:** `23_json_in_sqlite.ipynb`.